In [2]:
import sys, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, TargetEncoder,
                                   FunctionTransformer)
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

random_state = 42
best_x = 7          # chosen in 02_preprocessing
data_dir = "/Users/shouryakulkarni/IDX-Exchange-Data-Science"
pd.set_option('display.max_columns', 50)

print("python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

python 3.12.12 | pandas 3.0.0 | numpy 2.4.2 | scikit-learn 1.8.0


In [3]:
clean = pd.read_csv(f"{data_dir}/cleaned_sfr.csv",
                    dtype={"PostalCode": str}, parse_dates=["CloseDate"])
print(clean.shape)

(107416, 20)


In [4]:
target = "ClosePrice"

numeric_features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSqFt",
    "YearBuilt", "Latitude", "Longitude", "GarageSpaces", "Stories",
]
skewed_features = ["LivingArea", "LotSqFt"]
other_numeric = [c for c in numeric_features if c not in skewed_features]
binary_features = ["PoolPrivateYN", "ViewYN", "FireplaceYN", "NewConstructionYN"]
low_card_categorical = ["CountyOrParish"]
high_card_categorical = ["City", "PostalCode"]

features = numeric_features + binary_features + low_card_categorical + high_card_categorical

In [ ]:
def fit_outlier_bounds(train, lo=0.005, hi=0.995):
    ppsf = train[target] / train["LivingArea"]
    return {
        "price_lo": train[target].quantile(lo), "price_hi": train[target].quantile(hi),
        "ppsf_lo": ppsf.quantile(lo),           "ppsf_hi": ppsf.quantile(hi),
    }

def apply_outlier_bounds(data, b):
    ppsf = data[target] / data["LivingArea"]
    keep = (data[target].between(b["price_lo"], b["price_hi"]) &
            ppsf.between(b["ppsf_lo"], b["ppsf_hi"]))
    return data[keep]

def make_preprocessor():
    skewed = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    low_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ])
    high_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target", TargetEncoder(target_type="continuous", random_state=random_state)),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("skew", skewed, skewed_features),
        ("num", numeric, other_numeric),
        ("bin", "passthrough", binary_features),
        ("low", low_card, low_card_categorical),
        ("high", high_card, high_card_categorical),
    ])

def evaluate(y_true, y_pred):
    pct_err = np.abs(y_pred - y_true) / y_true
    return {
        "R2": r2_score(y_true, y_pred),
        "MAPE %": pct_err.mean() * 100,
        "MdAPE %": np.median(pct_err) * 100,
        "MAE $": mean_absolute_error(y_true, y_pred),
        "Within 20 %": (pct_err <= 0.20).mean() * 100,
    }

In [6]:
def fit_and_score(estimator, train, eval_set):
    """Learn outlier cutoffs on train, filter both sets, fit on log(price), score eval_set."""
    b = fit_outlier_bounds(train)
    tr, ev = apply_outlier_bounds(train, b), apply_outlier_bounds(eval_set, b)
    model = Pipeline([("prep", make_preprocessor()), ("model", clone(estimator))])
    model.fit(tr[features], np.log(tr[target]))
    pred = np.exp(model.predict(ev[features]))
    return evaluate(ev[target].values, pred), model, ev, pred

In [7]:
months = sorted(clean["CloseMonth"].unique())
test_month = months[-1]
val_month = months[-2]

# train on the 7 months before March, score on March
tune_train = clean[clean["CloseMonth"].isin(months[-2 - best_x : -2])]
val = clean[clean["CloseMonth"] == val_month]

# train on the 7 months before April, score on April (once)
final_train = clean[clean["CloseMonth"].isin(months[-1 - best_x : -1])]
test = clean[clean["CloseMonth"] == test_month]

print("Tuning:  train", months[-2 - best_x], "to", months[-3], f"({len(tune_train):,})",
      "| validate", val_month, f"({len(val):,})")
print("Final:   train", months[-1 - best_x], "to", months[-2], f"({len(final_train):,})",
      "| test", test_month, f"({len(test):,})")

Tuning:  train 2025-08 to 2026-02 (71,349) | validate 2026-03 (11,541)
Final:   train 2025-09 to 2026-03 (71,484) | test 2026-04 (12,455)


In [8]:
lr_val, _, _, _ = fit_and_score(LinearRegression(), tune_train, val)
pd.Series(lr_val, name="Linear Regression (validation)").round(3)

R2                  0.800
MAPE %             15.391
MdAPE %            11.813
MAE $          204797.950
Within 20 %        73.914
Name: Linear Regression (validation), dtype: float64

In [9]:
tree_rows = []
for depth in [6, 10, 14, 18, None]:
    for leaf in [5, 20, 50]:
        est = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf,
                                    random_state=random_state)
        m, _, _, _ = fit_and_score(est, tune_train, val)
        tree_rows.append({"max_depth": depth, "min_samples_leaf": leaf, **m})

tree_results = pd.DataFrame(tree_rows).sort_values("MdAPE %").round(3)
tree_results.head(10)

,max_depth,min_samples_leaf,R2,MAPE %,MdAPE %,MAE $,Within 20 %
10,18.0,20,0.838,14.408,10.280,193308.395,76.549
13,NaN,20,0.838,14.407,10.284,193290.115,76.540
6,14.0,5,0.820,14.774,10.440,199715.717,75.879
9,18.0,5,0.815,15.010,10.488,202294.681,75.015
7,14.0,20,0.838,14.513,10.516,194188.513,76.302
11,18.0,50,0.829,14.749,10.583,197834.660,75.562
14,NaN,50,0.829,14.749,10.584,197826.049,75.571
8,14.0,50,0.829,14.767,10.696,198002.088,75.535
12,NaN,5,0.815,15.297,10.863,205013.115,74.213
5,10.0,50,0.826,15.225,11.175,202749.728,74.178


In [10]:
best_tree = tree_results.iloc[0]
tree_params = {"max_depth": None if pd.isna(best_tree["max_depth"]) else int(best_tree["max_depth"]),
               "min_samples_leaf": int(best_tree["min_samples_leaf"])}
print("Best decision tree settings:", tree_params)

Best decision tree settings: {'max_depth': 18, 'min_samples_leaf': 20}


In [11]:
rf_rows = []
for leaf in [2, 5, 10]:
    for max_feat in [0.33, 0.6]:
        start = time.time()
        est = RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf,
                                    max_features=max_feat, n_jobs=-1,
                                    random_state=random_state)
        m, _, _, _ = fit_and_score(est, tune_train, val)
        rf_rows.append({"min_samples_leaf": leaf, "max_features": max_feat,
                        "seconds": round(time.time() - start, 1), **m})
        print(f"leaf={leaf}, max_features={max_feat}: MdAPE {m['MdAPE %']:.2f}%  ({rf_rows[-1]['seconds']}s)")

rf_results = pd.DataFrame(rf_rows).sort_values("MdAPE %").round(3)
rf_results

leaf=2, max_features=0.33: MdAPE 8.22%  (4.0s)
leaf=2, max_features=0.6: MdAPE 8.21%  (6.9s)
leaf=5, max_features=0.33: MdAPE 8.38%  (3.9s)
leaf=5, max_features=0.6: MdAPE 8.36%  (6.2s)
leaf=10, max_features=0.33: MdAPE 8.69%  (3.5s)
leaf=10, max_features=0.6: MdAPE 8.71%  (5.8s)


,min_samples_leaf,max_features,seconds,R2,MAPE %,MdAPE %,MAE $,Within 20 %
1,2,0.60,6.9,0.877,11.944,8.208,163133.465,82.780
0,2,0.33,4.0,0.877,11.874,8.215,162717.493,82.683
3,5,0.60,6.2,0.873,12.117,8.362,165784.934,82.392
2,5,0.33,3.9,0.871,12.134,8.385,166486.131,82.101
4,10,0.33,3.5,0.862,12.499,8.695,171539.818,81.140
5,10,0.60,5.8,0.866,12.471,8.706,170322.981,81.405


In [12]:
best_rf = rf_results.iloc[0]
rf_params = {"min_samples_leaf": int(best_rf["min_samples_leaf"]),
             "max_features": float(best_rf["max_features"])}
print("Best random forest settings:", rf_params)

Best random forest settings: {'min_samples_leaf': 2, 'max_features': 0.6}


In [13]:
final_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(**tree_params, random_state=random_state),
    "Random Forest": RandomForestRegressor(n_estimators=200, **rf_params, n_jobs=-1,
                                           random_state=random_state),
}

val_rows, test_rows, fitted, test_preds = {}, {}, {}, {}
for name, est in final_models.items():
    val_rows[name], _, _, _ = fit_and_score(est, tune_train, val)
    test_rows[name], fitted[name], test_f, test_preds[name] = fit_and_score(est, final_train, test)
    print("done:", name)

comparison = pd.concat({
    "validation (Mar)": pd.DataFrame(val_rows).T,
    "test (Apr)": pd.DataFrame(test_rows).T,
}).round(3)
comparison

done: Linear Regression
done: Decision Tree
done: Random Forest


R2  MAPE %  MdAPE %       MAE $  \
validation (Mar) Linear Regression  0.800  15.391   11.813  204797.950   
                 Decision Tree      0.838  14.408   10.280  193308.395   
                 Random Forest      0.878  11.899    8.215  162538.385   
test (Apr)       Linear Regression  0.813  15.601   11.699  213024.688   
                 Decision Tree      0.833  14.869   10.519  204779.482   
                 Random Forest      0.872  12.143    8.338  170346.306   

                                    Within 20 %  
validation (Mar) Linear Regression       73.914  
                 Decision Tree           76.549  
                 Random Forest           82.973  
test (Apr)       Linear Regression       73.414  
                 Decision Tree           75.656  
                 Random Forest           82.890

In [14]:
b = fit_outlier_bounds(final_train)
train_f = apply_outlier_bounds(final_train, b)
fit_rows = {}
for name, model in fitted.items():
    train_pred = np.exp(model.predict(train_f[features]))
    fit_rows[name] = {
        "train MdAPE %": evaluate(train_f[target].values, train_pred)["MdAPE %"],
        "test MdAPE %": test_rows[name]["MdAPE %"],
    }
overfit = pd.DataFrame(fit_rows).T
overfit["gap"] = overfit["test MdAPE %"] - overfit["train MdAPE %"]
overfit.round(2)

,train MdAPE %,test MdAPE %,gap
Linear Regression,11.41,11.70,0.29
Decision Tree,8.81,10.52,1.71
Random Forest,4.46,8.34,3.88


In [15]:
bands = pd.qcut(test_f[target], 5, labels=["Q1 cheapest", "Q2", "Q3", "Q4", "Q5 priciest"])
band_rows = []
for name, pred in test_preds.items():
    pct = (pred - test_f[target].values) / test_f[target].values * 100
    tmp = pd.DataFrame({"band": bands.values, "abs": np.abs(pct), "signed": pct})
    g = tmp.groupby("band", observed=True).agg(MdAPE=("abs", "median"), bias=("signed", "median"))
    g["model"] = name
    band_rows.append(g.reset_index())

band_table = pd.concat(band_rows).pivot(index="band", columns="model", values=["MdAPE", "bias"])
band_table.round(1)

MdAPE                                          bias  \
model       Decision Tree Linear Regression Random Forest Decision Tree   
band                                                                      
Q1 cheapest           9.4              11.4           7.4           2.7   
Q2                    8.3               9.8           6.2           0.1   
Q3                    8.9               9.9           7.0          -1.4   
Q4                   11.5              11.8           9.5          -4.2   
Q5 priciest          16.5              16.7          13.5         -10.5   

                                             
model       Linear Regression Random Forest  
band                                         
Q1 cheapest               4.3           1.7  
Q2                       -0.1          -1.0  
Q3                       -0.1          -1.7  
Q4                       -4.4          -4.2  
Q5 priciest             -13.0          -9.7

In [16]:
def backtest(estimator, data, X, n_cutoffs=3):
    months = sorted(data["CloseMonth"].unique())
    rows = []
    for tm in months[::-1][:n_cutoffs]:
        t = months.index(tm)
        if t < X:
            print(f"Skipping {tm}: needs {X} prior months, only {t} available")
            continue
        tr = data[data["CloseMonth"].isin(months[t - X:t])]
        te = data[data["CloseMonth"] == tm]
        m, _, _, _ = fit_and_score(estimator, tr, te)
        rows.append({"test month": tm, **m})
    return pd.DataFrame(rows)

bt = pd.concat({name: backtest(est, clean, best_x) for name, est in final_models.items()})
bt.round(3)

test month     R2  MAPE %  MdAPE %       MAE $  \
Linear Regression 0    2026-04  0.813  15.601   11.699  213024.688   
                  1    2026-03  0.800  15.391   11.813  204797.950   
                  2    2026-02  0.798  15.720   11.663  209351.074   
Decision Tree     0    2026-04  0.833  14.869   10.519  204779.482   
                  1    2026-03  0.838  14.408   10.280  193308.395   
                  2    2026-02  0.838  14.671   10.344  195887.538   
Random Forest     0    2026-04  0.872  12.143    8.338  170346.306   
                  1    2026-03  0.878  11.899    8.215  162538.385   
                  2    2026-02  0.878  12.178    8.454  164004.777   

                     Within 20 %  
Linear Regression 0       73.414  
                  1       73.914  
                  2       73.853  
Decision Tree     0       75.656  
                  1       76.549  
                  2       75.625  
Random Forest     0       82.890  
                  1       82.973  
                  2       81.520

In [17]:
bt_summary = bt.groupby(level=0)["MdAPE %"].agg(["mean", "min", "max"]).round(2)
bt_summary["range"] = bt_summary["max"] - bt_summary["min"]
bt_summary

,mean,min,max,range
Decision Tree,10.38,10.28,10.52,0.24
Linear Regression,11.72,11.66,11.81,0.15
Random Forest,8.34,8.22,8.45,0.23


In [18]:
rf_model = fitted["Random Forest"]
names = rf_model.named_steps["prep"].get_feature_names_out()
importance = pd.Series(rf_model.named_steps["model"].feature_importances_, index=names)
importance.sort_values(ascending=False).head(15).round(3)

high__PostalCode                   0.504
high__City                         0.202
skew__LivingArea                   0.156
num__BathroomsTotalInteger         0.044
num__Longitude                     0.020
skew__LotSqFt                      0.017
num__Latitude                      0.015
num__YearBuilt                     0.013
num__BedroomsTotal                 0.010
num__GarageSpaces                  0.003
low__CountyOrParish_Riverside      0.002
low__CountyOrParish_Los Angeles    0.001
bin__PoolPrivateYN                 0.001
num__Stories                       0.001
bin__ViewYN                        0.001
dtype: float64

In [19]:
summary = comparison.reset_index().rename(columns={"level_0": "split", "level_1": "model"})
summary["X"] = best_x
summary.to_csv(f"{data_dir}/model_comparison.csv", index=False)
print("Saved model_comparison.csv")

Saved model_comparison.csv


## Week 5 Findings: Decision Tree & Random Forest vs. Baseline

**Setup:** same features, preprocessing and outlier rules as the baseline. Settings tuned on March (validation), then retrained on Sep–Mar and tested once on April.
- Decision tree: max_depth = 18, min_samples_leaf = 20
- Random forest: 200 trees, min_samples_leaf = 2, max_features = 0.6

**Test results (April 2026)**
| Model | R² | MdAPE | Within 20% |
|---|---|---|---|
| Linear Regression | 0.813 | 11.70% | 73.4% |
| Decision Tree | 0.833 | 10.52% | 75.7% |
| **Random Forest** | **0.872** | **8.34%** | **82.9%** |

**Backtest (Feb–Apr):** RF MdAPE 8.22–8.45% vs. LR 11.66–11.81%. The RF's worst month beats the LR's best by 3+ points, so the improvement is real, not month-to-month noise (~0.2 pts).

**Strengths and weaknesses**
- **Linear Regression:** simple, stable, easy to interpret. Too rigid: one formula for all of California, so it underfits (train ≈ test error) and pulls predictions toward the middle (−13% on the priciest homes).
- **Decision Tree:** captures patterns a straight line can't (e.g., size matters differently in different areas). A single tree is unstable and memorizes (train 8.8% vs. test 10.5%).
- **Random Forest:** averaging many varied trees cancels individual errors. Best in every price band. Trade-offs: slower, much larger to store, harder to explain, and it memorizes training data (train 4.5% vs. test 8.3%), but settings were chosen on unseen data and test performance is far better.

**Remaining weakness:** still underestimates the most expensive homes by ~10% (vs. 13% for LR). Trees average similar training homes, and there are few luxury examples to average.

**What drives price:** location dominates (ZIP, City, lat/long ≈ 3/4 of importance), then living area.

**Next:** week 6 adds school-district location features; week 7 tries gradient boosting. Both target the luxury-home gap.